# HITS Algorithm — Hubs and Authorities

## Learning Objectives

By the end of this notebook you will be able to:

1. **Explain** the hub/authority duality and the mutual reinforcement principle
2. **Define** the HITS update rules and their matrix form
3. **Prove** that HITS converges to the principal eigenvectors of $AA^\top$ and $A^\top A$
4. **Distinguish** HITS from PageRank: query-specific vs global ranking
5. **Implement** HITS via alternating power iteration


## Problem Statement

### Limitation of PageRank

PageRank computes a *global* ranking of all pages. For a specific query (e.g. "best machine learning courses"), global importance may not match query relevance.

### HITS Insight — Two Roles for Web Pages

**Authorities** — pages that are genuine sources of information on a topic.  
**Hubs** — pages that collect and curate links to good authorities (e.g. "best of" lists, resource pages).

These two roles *mutually reinforce* each other:

- A good hub is one that links to many good authorities
- A good authority is one that is linked to by many good hubs

### Algorithm Scope

HITS operates on a **query-specific subgraph** $S_q$ expanded from the top-$k$ pages returned by a text search engine:

1. Start with root set $R$ = top 200 pages matching query $q$
2. Expand to base set $S$ by adding every page that $R$ links to, and every page that links to $R$ (bounded to ~5,000 pages)
3. Run HITS on $S$


In [ ]:
from IPython.display import SVG, display

svg = '''
<svg xmlns="http://www.w3.org/2000/svg" width="820" height="390" font-family="monospace" font-size="12">
  <rect width="820" height="390" fill="#fafafa" rx="8"/>
  <defs>
    <marker id="arr" markerWidth="9" markerHeight="7" refX="8" refY="3.5" orient="auto">
      <polygon points="0 0,9 3.5,0 7" fill="#666"/>
    </marker>
  </defs>

  <!-- ═══ LEFT: Graph ═══ -->
  <text x="185" y="22" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">Directed Graph</text>

  <!-- Hubs (orange rectangles) -->
  <rect x="30"  y="60"  width="80" height="34" rx="5" fill="#f28e2b"/>
  <rect x="30"  y="120" width="80" height="34" rx="5" fill="#f28e2b"/>
  <rect x="30"  y="180" width="80" height="34" rx="5" fill="#f28e2b"/>
  <text x="70" y="82"  text-anchor="middle" fill="white" font-size="11">Hub 1  h=0.57</text>
  <text x="70" y="142" text-anchor="middle" fill="white" font-size="11">Hub 2  h=0.71</text>
  <text x="70" y="202" text-anchor="middle" fill="white" font-size="11">Hub 3  h=0.41</text>

  <!-- Authorities (blue circles) -->
  <circle cx="290" cy="80"  r="26" fill="#4e79a7"/>
  <circle cx="290" cy="155" r="26" fill="#4e79a7"/>
  <circle cx="290" cy="230" r="26" fill="#4e79a7"/>
  <text x="290" y="76"  text-anchor="middle" fill="white" font-size="11">Auth 1</text>
  <text x="290" y="88"  text-anchor="middle" fill="white" font-size="10">a=0.67</text>
  <text x="290" y="151" text-anchor="middle" fill="white" font-size="11">Auth 2</text>
  <text x="290" y="163" text-anchor="middle" fill="white" font-size="10">a=0.59</text>
  <text x="290" y="226" text-anchor="middle" fill="white" font-size="11">Auth 3</text>
  <text x="290" y="238" text-anchor="middle" fill="white" font-size="10">a=0.45</text>

  <!-- Edges Hub1 -> Auth1, Auth2 -->
  <line x1="110" y1="75"  x2="264" y2="78"  stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="110" y1="82"  x2="264" y2="148" stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- Hub2 -> Auth1, Auth2, Auth3 -->
  <line x1="110" y1="135" x2="264" y2="83"  stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="110" y1="137" x2="264" y2="153" stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="110" y1="140" x2="264" y2="222" stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- Hub3 -> Auth2, Auth3 -->
  <line x1="110" y1="193" x2="264" y2="161" stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="110" y1="198" x2="264" y2="222" stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>

  <!-- legend -->
  <rect x="30" y="250" width="14" height="14" fill="#f28e2b" rx="2"/>
  <text x="50" y="262" fill="#555" font-size="11">Hub — points to good authorities</text>
  <circle cx="37" cy="278" r="7" fill="#4e79a7"/>
  <text x="50" y="282" fill="#555" font-size="11">Authority — pointed to by good hubs</text>

  <!-- ═══ RIGHT: Mutual Reinforcement ═══ -->
  <text x="580" y="22" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">Mutual Reinforcement (Update Rules)</text>

  <!-- Hub update box -->
  <rect x="420" y="40" width="360" height="70" rx="6" fill="#fff4e0" stroke="#f28e2b" stroke-width="1.5"/>
  <text x="600" y="63" text-anchor="middle" fill="#f28e2b" font-size="12" font-weight="bold">Hub Update</text>
  <text x="600" y="83" text-anchor="middle" fill="#333" font-size="13">h(v) ← Σ  a(u)  for each  v → u</text>
  <text x="600" y="100" text-anchor="middle" fill="#555" font-size="11">hub score = sum of authority scores it points to</text>

  <!-- Authority update box -->
  <rect x="420" y="130" width="360" height="70" rx="6" fill="#e8f0fb" stroke="#4e79a7" stroke-width="1.5"/>
  <text x="600" y="153" text-anchor="middle" fill="#4e79a7" font-size="12" font-weight="bold">Authority Update</text>
  <text x="600" y="173" text-anchor="middle" fill="#333" font-size="13">a(v) ← Σ  h(u)  for each  u → v</text>
  <text x="600" y="190" text-anchor="middle" fill="#555" font-size="11">authority score = sum of hub scores pointing to it</text>

  <!-- Matrix form -->
  <rect x="420" y="220" width="360" height="100" rx="6" fill="#f0fff0" stroke="#59a14f" stroke-width="1.5"/>
  <text x="600" y="243" text-anchor="middle" fill="#59a14f" font-size="12" font-weight="bold">Matrix Form</text>
  <text x="600" y="265" text-anchor="middle" fill="#333" font-size="12">h ← A · a  ≡  A · (Aᵀ · h)  =  AAᵀ h</text>
  <text x="600" y="285" text-anchor="middle" fill="#333" font-size="12">a ← Aᵀ · h  ≡  Aᵀ · (A · a)  =  AᵀA a</text>
  <text x="600" y="308" text-anchor="middle" fill="#555" font-size="11">h* = principal eigenvector of AAᵀ</text>
  <text x="600" y="323" text-anchor="middle" fill="#555" font-size="11">a* = principal eigenvector of AᵀA</text>

  <!-- Normalisation note -->
  <text x="600" y="355" text-anchor="middle" fill="#555" font-size="11">After each update: normalise so Σh² = 1 and Σa² = 1</text>

</svg>
'''

display(SVG(svg))


## The Adjacency Matrix

Let $V = \{v_1, \ldots, v_n\}$ be the nodes in the base set. Define the **adjacency matrix** $A \in \{0,1\}^{n \times n}$:

$$A_{ij} = \begin{cases} 1 & \text{if there is a directed edge } v_j \to v_i \\ 0 & \text{otherwise} \end{cases}$$

Note: $A_{ij} = 1$ means **column $j$ links to row $i$**, which is the standard "from column to row" convention used in link analysis.


## Derivation of the HITS Update Rules

Let $h \in \mathbb{R}^n$ be the hub score vector and $a \in \mathbb{R}^n$ be the authority score vector.

### Mutual Reinforcement

**Authority update:** A node $v_i$'s authority score is the sum of hub scores of pages pointing to it:

$$a_i \leftarrow \sum_{j: v_j \to v_i} h_j = \sum_j A_{ij}\, h_j \quad \Rightarrow \quad a \leftarrow A^\top h$$

**Hub update:** A node $v_j$'s hub score is the sum of authority scores of pages it points to:

$$h_j \leftarrow \sum_{i: v_j \to v_i} a_i = \sum_i A_{ij}\, a_i \quad \Rightarrow \quad h \leftarrow A\, a$$

### Substituting One Into the Other

Substitute $a \leftarrow A^\top h$ into $h \leftarrow A a$:

$$h \leftarrow A A^\top h$$

Substitute $h \leftarrow A a$ into $a \leftarrow A^\top h$:

$$a \leftarrow A^\top A\, a$$

### Convergence

Both are **power iteration** on the matrices $AA^\top$ and $A^\top A$:

- $h^*$ = **principal eigenvector** of $AA^\top$ (largest eigenvalue)
- $a^*$ = **principal eigenvector** of $A^\top A$ (largest eigenvalue)

Since $AA^\top$ and $A^\top A$ are positive semi-definite, both eigenvectors are real and non-negative. Power iteration converges to them provided no ties in the largest eigenvalue.

### Normalisation

After each step, normalise to prevent divergence:

$$h \leftarrow \frac{h}{\|h\|_2}, \quad a \leftarrow \frac{a}{\|a\|_2}$$

Normalisation does not affect the direction of convergence (the eigenvector is defined up to scale).


## Algorithm Steps

### Inputs

- Directed graph adjacency list for base set $S$
- Number of iterations (or convergence tolerance)

### Steps

1. **Initialise** $h = \mathbf{1}$, $a = \mathbf{1}$

2. **Iterate** until convergence:

   a. $a \leftarrow A^\top h$ — authority update (sum hub scores of in-neighbours)

   b. $h \leftarrow A\, a$ — hub update (sum authority scores of out-neighbours)

   c. Normalise: $h \leftarrow h / \|h\|_2$, $a \leftarrow a / \|a\|_2$

3. **Output** ranked authorities (sort by $a^*$) and ranked hubs (sort by $h^*$)

### Complexity

- Per iteration: $O(|E|)$ for sparse adjacency multiplication
- Typically converges in 20–50 iterations

### HITS vs PageRank

| | HITS | PageRank |
|---|------|----------|
| **Scope** | Query-specific subgraph | Global web graph |
| **Output** | Two vectors: hub & authority | One vector: importance |
| **Manipulation** | Vulnerable to link spam | More robust with teleportation |
| **Use case** | Topic-specific authority ranking | Global web ranking |


In [ ]:
import numpy as np
from collections import defaultdict


def hits(adj, max_iter=100, tol=1e-8):
    """
    Compute HITS hub and authority scores via power iteration.

    Inputs
    ------
    adj     : dict {node: list_of_neighbours} — directed out-links
    max_iter: int — maximum iterations
    tol     : float — L2 norm convergence threshold

    Output
    ------
    hubs        : dict {node: hub_score}
    authorities : dict {node: authority_score}
    n_iters     : int
    """
    nodes = sorted(set(adj.keys()) | {v for nbrs in adj.values() for v in nbrs})
    n = len(nodes)
    idx = {node: i for i, node in enumerate(nodes)}

    # Build adjacency matrix A[i,j] = 1 iff edge j->i exists
    A = np.zeros((n, n))
    for src, nbrs in adj.items():
        for dst in nbrs:
            A[idx[dst], idx[src]] = 1.0

    # Initialise hub and authority vectors
    h = np.ones(n)
    a = np.ones(n)

    for i in range(max_iter):
        # Authority update: a ← Aᵀ h (each node gets sum of hub scores pointing to it)
        a_new = A.T @ h
        # Hub update: h ← A a (each node gets sum of authority scores it points to)
        h_new = A @ a_new

        # Normalise to unit L2 norm
        a_new /= np.linalg.norm(a_new) or 1.0
        h_new /= np.linalg.norm(h_new) or 1.0

        if np.linalg.norm(h_new - h) + np.linalg.norm(a_new - a) < tol:
            h, a = h_new, a_new
            return (
                {nodes[j]: float(h[j]) for j in range(n)},
                {nodes[j]: float(a[j]) for j in range(n)},
                i + 1,
            )
        h, a = h_new, a_new

    return (
        {nodes[j]: float(h[j]) for j in range(n)},
        {nodes[j]: float(a[j]) for j in range(n)},
        max_iter,
    )


# ── Demo ──────────────────────────────────────────────────────────────────────
# Simulate a query subgraph: hub pages linking to authoritative pages
adj = {
    "hub_A":   ["auth_1", "auth_2", "auth_3"],
    "hub_B":   ["auth_1", "auth_2"],
    "hub_C":   ["auth_2", "auth_3"],
    "auth_1":  ["hub_A"],        # authority pointing back (creates cycles)
    "auth_2":  [],
    "auth_3":  ["auth_2"],
}

hubs, auths, iters = hits(adj)
print(f"Converged in {iters} iterations\n")

print("Hub scores (higher = better hub):")
for node, score in sorted(hubs.items(), key=lambda x: -x[1]):
    bar = '█' * int(score * 40)
    print(f"  {node:<12s}: {score:.4f}  {bar}")

print("\nAuthority scores (higher = better authority):")
for node, score in sorted(auths.items(), key=lambda x: -x[1]):
    bar = '█' * int(score * 40)
    print(f"  {node:<12s}: {score:.4f}  {bar}")
